### Structured output

Model can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. Langchain supports multiple schema types and methods for enforcing structured output.

### Pydantic

Pydantic models provide the richest feature set with field validation, descriptions and nested structures.

In [1]:
import os
from langchain.chat_models import init_chat_model
os.environ["MISTRAL_API_KEY"]=os.getenv("MISTRAL_API_KEY")
model = init_chat_model("mistral-small-latest")
model

ChatMistralAI(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14', 'langchain-mistralai': '1.1.6'}}, output_version=None, profile={'name': 'Mistral Small (latest)', 'release_date': '2026-03-16', 'last_updated': '2026-03-16', 'open_weights': True, 'max_input_tokens': 256000, 'max_output_tokens': 256000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': True, 'temperature': True}, client=<httpx.Client object at 0x00000200070EB4D0>, async_client=<httpx.AsyncClient object at 0x0000020007239550>, mistral_api_key=SecretStr('**********'), endpoint='https://api.mistral.ai/v1', model='mistral-small-latest', model_kwargs={})

In [7]:
from pydantic import BaseModel, Field
class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movies rating out of 10")

In [8]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatMistralAI(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14', 'langchain-mistralai': '1.1.6'}}, output_version=None, profile={'name': 'Mistral Small (latest)', 'release_date': '2026-03-16', 'last_updated': '2026-03-16', 'open_weights': True, 'max_input_tokens': 256000, 'max_output_tokens': 256000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': True, 'temperature': True}, client=<httpx.Client object at 0x00000237BBF77770>, async_client=<httpx.AsyncClient object at 0x00000237BBF778C0>, mistral_api_key=SecretStr('**********'), endpoint='https://api.mistral.ai/v1', model='mistral-small-latest', model_kwargs={}), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'descr

In [9]:
model.invoke("Provide details about the movie Inception.")

AIMessage(content='Certainly! *Inception* (2010) is a science fiction heist film written and directed by **Christopher Nolan**. Known for its intricate plot, stunning visuals, and mind-bending concepts, the movie explores the depths of dreams, reality, and human consciousness.\n\n### **Basic Details:**\n- **Release Date:** July 16, 2010 (USA)\n- **Runtime:** 148 minutes\n- **Genre:** Science Fiction, Action, Thriller\n- **Director:** Christopher Nolan\n- **Screenplay:** Christopher Nolan\n- **Music:** Hans Zimmer\n- **Cinematography:** Wally Pfister\n- **Production Company:** Warner Bros., Legendary Pictures, Syncopy\n\n### **Plot Summary:**\nThe story follows **Dom Cobb (Leonardo DiCaprio)**, a skilled "extractor" who steals secrets from people’s subconscious through shared dreaming. Haunted by the death of his wife, **Mal (Marion Cotillard)**, Cobb is offered a chance at redemption by businessman **Saito (Ken Watanabe)**—if he can perform "inception," the nearly impossible task of pl

In [10]:
response=model_with_structure.invoke("Provide details about the movie Inception")
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

### Message output alongside parsed structure

In [12]:
from pydantic import BaseModel, Field
class Movie(BaseModel):
    """A movie with details."""
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movies rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)

response = model_with_structure.invoke("Provide details about the movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'FjOSCSLgZ', 'type': 'function', 'function': {'name': 'Movie', 'arguments': '{"title": "Inception", "year": 2010, "director": "Christopher Nolan", "rating": 8.8}'}, 'index': 0}]}, response_metadata={'token_usage': {'prompt_tokens': 159, 'total_tokens': 194, 'completion_tokens': 35, 'prompt_tokens_details': {'cached_tokens': 0}, 'service_tier': 'standard'}, 'model_name': 'mistral-small-latest', 'model': 'mistral-small-latest', 'finish_reason': 'tool_calls', 'model_provider': 'mistralai'}, id='lc_run--01a00695-6e1d-7840-8cad-83a018095123-0', tool_calls=[{'name': 'Movie', 'args': {'title': 'Inception', 'year': 2010, 'director': 'Christopher Nolan', 'rating': 8.8}, 'id': 'FjOSCSLgZ', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 159, 'output_tokens': 35, 'total_tokens': 194}),
 'parsed': Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8),
 'parsing_error': None}

### Nested Structure

In [13]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name : str
    role : str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito')], genres=['Action', 'Sci-Fi', 'Thriller'], budget=160.0)

### TypeDict

TypeDict provides a simpler alternative using Python's build in typing, ideal when you don't need runtime validation.

In [3]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year when movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]

model_with_typedict = model.with_structured_output(MovieDict)
response = model_with_typedict.invoke("Provide the details of the movie Avengers : The End Game")
response

{'title': 'Avengers: The End Game',
 'year': 2019,
 'director': 'Joe Russo',
 'rating': 8.9}

In [6]:

class Actor(TypedDict):
    name : str
    role : str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Inception")
response

{'title': 'Inception',
 'year': 2010,
 'cast': [{'name': 'Leonardo DiCaprio', 'role': 'Dom Cobb'},
  {'name': 'Joseph Gordon-Levitt', 'role': 'Arthur'},
  {'name': 'Elliot Page', 'role': 'Ariadne'},
  {'name': 'Tom Hardy', 'role': 'Eames'},
  {'name': 'Ken Watanabe', 'role': 'Saito'},
  {'name': 'Cillian Murphy', 'role': 'Robert Fischer'}],
 'genres': ['Action', 'Sci-Fi', 'Thriller'],
 'budget': 160000000}

In [8]:
model.profile


{'name': 'Mistral Small (latest)',
 'release_date': '2026-03-16',
 'last_updated': '2026-03-16',
 'open_weights': True,
 'max_input_tokens': 256000,
 'max_output_tokens': 256000,
 'text_inputs': True,
 'image_inputs': True,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'attachment': True,
 'temperature': True}

### Data Classes

A data class is a class typically containing mainly data, although there aren't really any restrictions. You create it using the @dataclass decorator

In [14]:
import os
os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")

In [ ]:
# from pydantic
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact Information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model = "google_genai:gemini-3.6-flash",
    response_format = ContactInfo
)

result = agent.invoke({
    "messages": [{"role": "user", "content":"Extract contact into from: John Doe, john@example.com, (+91) 7209587201"}]
})

result

{'messages': [HumanMessage(content='Extract contact into from: John Doe, john@example.com, (+91) 7209587201', additional_kwargs={}, response_metadata={}, id='0e026885-1350-4fd2-850b-341cb85cb6e9'),
  AIMessage(content=[{'type': 'text', 'text': '{"name":"John Doe","email":"john@example.com","phone":"(+91) 7209587201"}', 'extras': {'signature': 'ErgJCrUJARFNMg/HaWlVtzqIvIWw+dIjbOW2GdNe41L9PiqF8SwJzsSpG37EIL5JLW1Nbb2mU1NF/yaNH+x5PH6PZqAxwmFQCcCVivlDPgOKGYnl6oydLIbg9EzYYw+i3oEyU6VF70L4dLnH1PAU25PE8o0k3c7qxd63I3e6TQvQ+6GSL2YLPcidbKEoGhQv2Y5iLcWlpLReyfzx7ZX96qFcY89UI0IRA/cPo/tS80RTEDMAYXFRNjm0uZNUMjbyM/2fwuJbpjBiyxckMhEZfNcKOK8aQKcaS5oyD3dXSoQWFiETreBz6cgU2UYrlvkP7me2zd766RBXKNSIa/Mxf4cwQQ/zE986ZwQN9uZbCDjqCUAcj4Le6KETuQuBIk0/Ij8V6KLzcEcociDTH7GkINLClx+af2pl5UuHIHjrLIvG5TTowa2LEbzonKsTueIyBwxh89+QAvy7x4FUnNtiXNVKcqZl2jFU0lX2UI2BwOsSwUYlBpiZ8D+YZ6k6du8J1x4DrAo4xHehkym2/3loA/umdBWaxhMJslhR4mWx/uDFxgvnTZAOJt5XJwP7RGympYOWziodjPrINVyrd6untzjJgMVaeS6P32Z0bR3On998abh1kDf+qvZbRQfKmaxWD+M6vxfdWldTnK

In [16]:
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(+91) 7209587201')

In [ ]:
# from Typedict
from typing_extensions import TypedDict
from langchain.agents import create_agent

class ContactInfo(TypedDict):
    """Contact Information for a person."""
    name: str 
    email: str 
    phone: str
    
agent = create_agent(
    model = "google_genai:gemini-3.6-flash",
    response_format = ContactInfo
)

result = agent.invoke({
    "messages": [{"role": "user", "content":"Extract contact into from: John Doe, john@example.com, (+91) 7209587201"}]
})

result["structured_response"]

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(+91) 7209587201'}

In [19]:
# from Dataclass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact Information for a person"""
    name: str
    email: str
    phone: str

agent = create_agent(
    model = "google_genai:gemini-3.6-flash",
    response_format = ContactInfo
)

result = agent.invoke({
    "messages": [{"role": "user", "content":"Extract contact into from: John Doe, john@example.com, (+91) 7209587201"}]
})

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(+91) 7209587201')